In [ ]:
import os
import librosa
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load metadata
metadata_path = "./archive/UrbanSound8K.csv"
metadata = pd.read_csv(metadata_path)

# Create file paths and labels
file_paths = ["./archive/fold{}/{}".format(row['fold'], row["slice_file_name"]) for _, row in metadata.iterrows()]
labels = metadata["classID"].values

# Split into train and test sets
train_file_paths, test_file_paths, y_train, y_test = train_test_split(file_paths, labels, test_size=0.2, random_state=42)

# Feature extraction function
def extract_features(file_path, n_mfcc=40, max_pad_len=174):
    try:
        audio, sample_rate = librosa.load(file_path, sr=22050)
        mfccs = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=n_mfcc)

        # Pad or truncate to fixed shape
        if mfccs.shape[1] < max_pad_len:
            pad_width = max_pad_len - mfccs.shape[1]
            mfccs = np.pad(mfccs, ((0, 0), (0, pad_width)), mode='constant')
        else:
            mfccs = mfccs[:, :max_pad_len]

        return mfccs.T  # Shape: (174, 40)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return None

# Apply feature extraction
X_train_features = np.array([f for f in (extract_features(fp) for fp in train_file_paths) if f is not None])
X_test_features = np.array([f for f in (extract_features(fp) for fp in test_file_paths) if f is not None])

# Normalize features
scaler = StandardScaler()
X_train_features = scaler.fit_transform(X_train_features.reshape(-1, X_train_features.shape[-1])).reshape(X_train_features.shape)
X_test_features = scaler.transform(X_test_features.reshape(-1, X_test_features.shape[-1])).reshape(X_test_features.shape)

# Ensure labels are one-hot encoded
y_train = to_categorical(y_train, num_classes=10)
y_test = to_categorical(y_test, num_classes=10)

# Transformer-based model
class TransformerBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, rate=0.1):
        super(TransformerBlock, self).__init__()
        self.att = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(ff_dim, activation="relu"),
            tf.keras.layers.Dense(embed_dim)
        ])
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, inputs, training):
        attn_output = self.att(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(inputs + attn_output)
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        return self.layernorm2(out1 + ffn_output)

# Model function
def build_transformer_model(input_shape, embed_dim=64, num_heads=4, ff_dim=128, num_classes=10):
    inputs = tf.keras.Input(shape=input_shape)
    
    # Linear embedding layer
    x = tf.keras.layers.Dense(embed_dim)(inputs)
    
    # Transformer blocks
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)
    x = TransformerBlock(embed_dim, num_heads, ff_dim)(x)

    # Global average pooling
    x = tf.keras.layers.GlobalAveragePooling1D()(x)

    # Dense layers
    x = tf.keras.layers.Dense(128, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    
    outputs = tf.keras.layers.Dense(num_classes, activation="softmax")(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="categorical_crossentropy", metrics=["accuracy"])
    
    return model

# Build the transformer model
input_shape = (X_train_features.shape[1], X_train_features.shape[2])
transformer_model = build_transformer_model(input_shape)

# Train the model
history = transformer_model.fit(X_train_features, y_train, validation_data=(X_test_features, y_test),
                                epochs=30, batch_size=32, verbose=1)

# Evaluate the model
score = transformer_model.evaluate(X_test_features, y_test)
print(f"Transformer Model Accuracy: {score[1] * 100:.2f}%")
